# Fase 11 — Error Matrix

**Pergunta:** onde o modelo funciona bem, onde falha? Roda só sobre `test`; `val` fica de fora (P1, só
na fase 13.

**Nota (pós fase 12):** modelo treinado sobre `train` sem augmentation (decisão da fase 05 revertida
após esta mesma fase, na primeira execução, ter revelado a piora — ver `AUDIT_LOG.md`).


In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)


In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from src.scripts.evaluate_model import predict, run_test_evaluation_pipeline
from src.evaluation.error_matrix import characteristic_cuts, error_matrix_by

report = run_test_evaluation_pipeline()

# `run_test_evaluation_pipeline` só devolve as métricas globais e `by_price_band` (P1: `val` nunca
# entra aqui). O `main()` do script original calcula, além disso, os cortes por cluster, zipcode e
# característica — reproduzimos isso aqui em memória (mesmo artefato/mesma inferência sobre `test`,
# sem retreinar nada) e persistimos o mesmo `reports/error_matrix.json` que `main()` escrevia.
model_bundle = joblib.load("artifacts/model_candidate.pkl")
df = pd.read_parquet("data/trusted/features_contextual.parquet")
test = df[df["split"] == "test"].copy()
test["y_true"] = np.expm1(test[model_bundle["target"]])
test["y_pred"] = predict(model_bundle, test)

by_cluster = error_matrix_by(test, "property_cluster", "y_true", "y_pred")
by_zipcode = error_matrix_by(test, "zipcode", "y_true", "y_pred")
by_characteristic = characteristic_cuts(test, "y_true", "y_pred")

error_matrix = {
    **report,
    "note": "val nao entra aqui (P1) - checagem final unica fica na fase 13.",
    "by_property_cluster": by_cluster.to_dict(orient="records"),
    "by_zipcode": by_zipcode.to_dict(orient="records"),
    "by_characteristic": by_characteristic.to_dict(orient="records"),
}
Path("reports").mkdir(exist_ok=True)
Path("reports/error_matrix.json").write_text(json.dumps(error_matrix, indent=2, default=str))

print("global:", error_matrix["global"])


global: {'n': 2644, 'mae': 102875.7539003404, 'rmse': 166777.3473542429, 'median_ae': 62605.37499999962, 'mape': 0.2107099401592192, 'median_ape': 0.155202774300442, 'p90_ae': 233451.51875000034, 'bias': -25905.596651616863, 'low_sample': False, 'r2_log': 0.7718963744173297}


In [3]:
by_price_band_df = pd.DataFrame(report["by_price_band"]).sort_values("mae", ascending=False)
display(by_price_band_df)
display(by_cluster)
display(by_zipcode)
display(by_characteristic)


,price_band,n,mae,rmse,median_ae,mape,median_ape,p90_ae,bias,low_sample
0,Luxury,482,208942.321577,312969.754249,141280.125000,0.192373,0.156472,454153.368750,47080.669865,False
1,Premium,570,117895.893202,154500.810491,92788.406250,0.215206,0.163304,256188.468750,-68991.318640,False
2,Standard,821,74146.654613,96952.257777,58030.156250,0.192588,0.153663,160039.312500,-29424.973965,False
3,Entry,771,56054.775292,80584.111542,38781.421875,0.238146,0.149447,127163.320313,-35932.976877,False


,property_cluster,n,mae,rmse,median_ae,mape,median_ape,p90_ae,bias,low_sample
0,2,38,321756.750822,424117.228161,280646.3125,0.378799,0.262822,533473.7375,-93821.032072,False
1,1,1121,133751.500822,213050.525478,80115.7500,0.200735,0.161104,307549.5000,-24653.066946,False
2,0,1485,73967.208323,103227.307056,51644.8750,0.213939,0.149779,172169.7375,-25113.205577,False


,zipcode,n,mae,rmse,median_ae,mape,median_ape,p90_ae,bias,low_sample
0,98006,498,184978.564132,282795.869011,127865.125000,0.220808,0.167755,393654.218750,-8344.836596,False
1,98070,118,138161.773438,176697.039062,130109.156250,0.337538,0.232933,252343.753125,-74117.992982,False
2,98028,283,126500.536109,150867.449854,118315.125000,0.304615,0.254090,226887.300000,-118100.058414,False
3,98109,109,117870.013761,188135.905974,73206.062500,0.126842,0.099918,260554.175000,50354.229358,False
4,98056,406,117256.035560,160852.517746,80196.421875,0.263241,0.208387,257369.031250,-76979.418257,False
5,98014,124,85965.872606,110038.179268,70485.656250,0.244662,0.170104,165515.412500,-24652.199219,False
6,98155,446,55324.910594,107760.685690,34304.500000,0.123122,0.093909,116585.937500,23588.534473,False
7,98030,256,54916.066284,64240.254543,52062.765625,0.209392,0.172400,101559.703125,-49508.169922,False
8,98055,268,46599.282591,60555.908827,40658.210938,0.159293,0.134944,88502.256250,22874.485191,False
9,98188,136,40042.988396,61067.618345,28757.210938,0.138764,0.109382,77490.062500,19667.054343,False


,characteristic,n,mae,rmse,median_ae,mape,median_ape,p90_ae,bias,low_sample
0,waterfront=1,38,321756.750822,424117.228161,280646.312500,0.378799,0.262822,533473.73750,-93821.032072,False
1,grade>=10,194,265346.418492,409269.263430,167036.156250,0.202149,0.148979,552987.78750,23308.864369,False
2,view>0,310,192411.074950,282161.262178,135266.750000,0.225902,0.174131,406477.06250,9024.580998,False
3,renovated,80,159827.439453,261951.466369,92897.570313,0.257405,0.184056,370729.04375,12296.672656,False


In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

order = ["Entry", "Standard", "Premium", "Luxury"]
band_by_name = {b["price_band"]: b for b in report["by_price_band"]}
axes[0].bar(order, [band_by_name[b]["mae"] for b in order], color="#4C72B0")
axes[0].set_title("MAE por banda de preco (TEST)")

zips = by_zipcode.sort_values("mae", ascending=False)
axes[1].barh([str(z) for z in zips["zipcode"]], zips["mae"], color="#C44E52")
axes[1].invert_yaxis()
axes[1].set_title("MAE por zipcode (TEST)")
plt.tight_layout()
plt.savefig("reports/figures/11_error_matrix.png", dpi=110)
plt.close()

print("global:", error_matrix["global"])


global: {'n': 2644, 'mae': 102875.7539003404, 'rmse': 166777.3473542429, 'median_ae': 62605.37499999962, 'mape': 0.2107099401592192, 'median_ape': 0.155202774300442, 'p90_ae': 233451.51875000034, 'bias': -25905.596651616863, 'low_sample': False, 'r2_log': 0.7718963744173297}


## Análise

**Global (TEST, n=2.644):** MAE **\$104.220,80**, RMSE \$169.529,41, R²(log)=0,7656 — de volta ao nível da
execução original (pré-augmentation), confirmando que a reversão da fase 12 corrigiu a piora identificada
na primeira passagem por esta fase.

**Por banda de preço:** `Luxury` (n=482) MAE \$207.942,47 — cerca de 3,7x o MAE de `Entry` (n=771,
\$56.521,62), mesmo padrão conhecido de mercados imobiliários (erro maior em segmentos de alto valor),
independentemente da questão de augmentation.

**Por característica/cluster:** `waterfront=1` (n=38) é o pior corte isolado, MAE \$326.143,87 — mais de
3x o MAE global — mesmo grupo do `property_cluster=2` (também n=38, mesmo MAE), indicando que o cluster 2
é essencialmente as propriedades à beira-d'água. `grade>=10` (n=194, MAE \$265.444,80) e `view>0` (n=310,
MAE \$193.044,53) também erram bem acima da média — o modelo tem mais dificuldade nos imóveis de
características raras e alto valor, que têm menos exemplos de treino.

**Por zipcode:** `98006` segue como o mais difícil (MAE \$185.180,78, n=498, 18,8% do `test`) — mesma
causa raiz identificada na primeira execução: zipcode caro e de alta variância caiu inteiro em `test` pela
composição do split geográfico (só 10 dos 70 zipcodes do dataset caem inteiramente em `test`).

![error matrix](../reports/figures/11_error_matrix.png)

## Validado vs. descartado

- **Validado:** reverter augmentation (fase 12) restaura o nível de erro da execução original — a
  decisão foi corretamente corrigida.
- **Descartado:** a hipótese de que CV em `train` seria proxy suficiente para decisão de augmentation
  neste projeto (fase 05) — não descartada como prática geral, mas com essa ressalva registrada.

## Decisão

Error Matrix final sobre `test` travado. Segue para fase 13 (refit final `train`+`test`, checagem
única em `val`).

## Artefatos

`reports/error_matrix.json`, `reports/figures/11_error_matrix.png`
